# DWBI Week 5 — The ETL Process

**Ali Raza** · `su92-bsdsm-f23-018` · **BSDS-7A** · Superior University

PakFreight's Week-2 merge rebuilt as a pipeline that reruns: 37 rows land, 35 append, 34 load — 578,650 kg and PKR 4,808,000, exactly. Every number below is computed from `data/source/`, `data/dims/` or `data/night2/` in the notebook itself; nothing is typed in as a constant except the assertion target it is checked against.

## Lab — Rebuild the Week 2 merge as a pipeline (Power Query guide, in pandas)

A notebook version of `PowerQuery_Lab_Guide.md`: the nine-step pipeline in
pandas, with the same logic as the Python track appendix. The applied-steps
pane is replaced by four things, exactly as the guide requires: a named `STEPS`
list keeping every intermediate frame, an explicit date format
(`format="%d/%m/%Y"`, never a bare `pd.to_datetime` and never `dayfirst=True`),
`drop_duplicates(subset=["ShipmentID"])` (never bare), and `how="left"` then
`fillna(-1)` (never `how="inner"`).

**Intermediate vs published:** `extract_*`, `renamed_*`, `trimmed_*` and
`appended` are intermediate frames — the staging surface, equivalent to
queries with **Enable load, off**. Only `fact` (and, in step 9, the keyed fact)
is published. Nothing else in this notebook is a table a user should see.

In [1]:

from pathlib import Path
import pandas as pd

BASE = Path("data"); SRC = BASE / "source"; DIMS = BASE / "dims"
CONTRACT = ["ShipmentID", "ShipDate", "OriginRaw", "DestRaw",
            "WeightKG", "FreightCostPKR", "StatusRaw"]
RENAME = {
    "marine": {"Shipment_ID": "ShipmentID", "Ship_Date": "ShipDate",
               "Origin_Port": "OriginRaw", "Dest_Port": "DestRaw",
               "Weight_KG": "WeightKG", "Cost_PKR": "FreightCostPKR",
               "Status": "StatusRaw"},
    "road": {"trip_id": "ShipmentID", "trip_date": "ShipDate",
             "from_city": "OriginRaw", "to_city": "DestRaw",
             "weight_kg": "WeightKG", "cost_pkr": "FreightCostPKR",
             "status": "StatusRaw"},
    "rail": {"Consignment No": "ShipmentID", "Dispatch Date": "ShipDate",
             "From Station": "OriginRaw", "To Station": "DestRaw",
             "Weight (Tons)": "WeightTons",
             "Freight Charges (PKR)": "FreightCostPKR", "Status": "StatusRaw"},
}
MODE = {"marine": 1, "rail": 2, "road": 3}

def extract(src, name):
    if name == "rail":
        return pd.read_excel(src / "rail_shipments.xlsx", sheet_name="Rail", skiprows=2)
    return pd.read_csv(src / f"{name}_shipments.csv")

def transform(df, name):
    df = df.rename(columns=RENAME[name])
    ids = df.ShipmentID.astype("string").str.strip()
    df = df[ids.notna() & (ids != "") & (ids.str.upper() != "TOTAL")].copy()
    for c in ("OriginRaw", "DestRaw"):
        df[c] = df[c].astype("string").str.strip()
    df["StatusRaw"] = (df.StatusRaw.astype("string")
                       .str.replace(r"\s+", " ", regex=True)
                       .str.strip().str.title())
    if name == "rail":
        df["ShipmentID"] = "RL-" + df.ShipmentID.astype(int).astype(str)
        df["WeightKG"] = df.WeightTons * 1000
        df = df.drop(columns=["WeightTons"])
        df["ShipDate"] = pd.to_datetime(df.ShipDate)
    else:
        df["ShipmentID"] = df.ShipmentID.astype("string").str.strip().str.upper()
        fmt = "%d/%m/%Y" if name == "marine" else "%Y-%m-%d"
        df["ShipDate"] = pd.to_datetime(df.ShipDate, format=fmt)
    df["WeightKG"] = df.WeightKG.astype("int64")
    df["FreightCostPKR"] = df.FreightCostPKR.astype("int64")
    df["ModeKey"] = MODE[name]
    return df[CONTRACT + ["ModeKey"]]

def alias_lookup():
    dim = pd.read_csv(DIMS / "DimCity.csv")
    out = (dim[["CityKey", "RawAliases"]]
           .assign(RawAliases=lambda d: d.RawAliases.str.split(" | ", regex=False))
           .explode("RawAliases"))
    out["RawAliases"] = out.RawAliases.str.strip()
    out = out[out.RawAliases.notna() & (out.RawAliases != "")]
    return out.rename(columns={"RawAliases": "RawSpelling"})

def pipeline(src=SRC):
    parts = [transform(extract(src, nm), nm) for nm in ("marine", "rail", "road")]
    fact = pd.concat(parts, ignore_index=True)
    appended = len(fact)
    fact = fact.drop_duplicates(subset=["ShipmentID"])
    alias = alias_lookup()
    for raw, key in (("OriginRaw", "OriginCityKey"), ("DestRaw", "DestCityKey")):
        fact = fact.merge(alias.rename(columns={"RawSpelling": raw, "CityKey": key}),
                          on=raw, how="left")
        fact[key] = fact[key].fillna(-1).astype("int64")
    return fact.reset_index(drop=True), appended, parts

# Lab step list: named steps, in run order, every intermediate frame kept for inspection
STEPS = ["Source", "SkippedTitleRows", "PromotedHeaders", "RenamedToContract",
         "RemovedBlankRows", "RemovedTotalRow", "TrimmedTextColumns",
         "NormalisedStatusCase", "PrefixedRailIDs", "ConvertedTonsToKG",
         "ParsedDatesUsingLocale", "AddedModeKey", "Appended",
         "RemovedDuplicates", "BuiltAliasLookup", "MergedOriginKey",
         "MergedDestKey", "ReplacedNullsWithMinusOne"]
frames = {}
raw = {nm: extract(SRC, nm) for nm in ("marine", "rail", "road")}
frames["Source"] = {k: v.copy() for k, v in raw.items()}
print("STEPS:", len(STEPS), "| step counts — Road 11, Marine 9, Rail 13, FactStaging 5 (guide §steps 1-7)")
print("landed rows:", {k: len(v) for k, v in raw.items()}, "-> total", sum(len(v) for v in raw.values()))
assert sum(len(v) for v in raw.values()) == 37

STEPS: 18 | step counts — Road 11, Marine 9, Rail 13, FactStaging 5 (guide §steps 1-7)
landed rows: {'marine': 12, 'rail': 10, 'road': 15} -> total 37


**Analysis.** The pipeline starts where Power Query starts: with the sources read faithfully and nothing corrected on the way in — 37 rows landed, junk included. Keeping every intermediate frame in `frames` is what makes a script inspectable the way the applied-steps pane makes a query inspectable: when step 9 is wrong, you look at step 7's frame instead of guessing.

In [2]:

from pathlib import Path
import pandas as pd

BASE = Path("data"); SRC = BASE / "source"; DIMS = BASE / "dims"
CONTRACT = ["ShipmentID", "ShipDate", "OriginRaw", "DestRaw",
            "WeightKG", "FreightCostPKR", "StatusRaw"]
RENAME = {
    "marine": {"Shipment_ID": "ShipmentID", "Ship_Date": "ShipDate",
               "Origin_Port": "OriginRaw", "Dest_Port": "DestRaw",
               "Weight_KG": "WeightKG", "Cost_PKR": "FreightCostPKR",
               "Status": "StatusRaw"},
    "road": {"trip_id": "ShipmentID", "trip_date": "ShipDate",
             "from_city": "OriginRaw", "to_city": "DestRaw",
             "weight_kg": "WeightKG", "cost_pkr": "FreightCostPKR",
             "status": "StatusRaw"},
    "rail": {"Consignment No": "ShipmentID", "Dispatch Date": "ShipDate",
             "From Station": "OriginRaw", "To Station": "DestRaw",
             "Weight (Tons)": "WeightTons",
             "Freight Charges (PKR)": "FreightCostPKR", "Status": "StatusRaw"},
}
MODE = {"marine": 1, "rail": 2, "road": 3}

def extract(src, name):
    if name == "rail":
        return pd.read_excel(src / "rail_shipments.xlsx", sheet_name="Rail", skiprows=2)
    return pd.read_csv(src / f"{name}_shipments.csv")

def transform(df, name):
    df = df.rename(columns=RENAME[name])
    ids = df.ShipmentID.astype("string").str.strip()
    df = df[ids.notna() & (ids != "") & (ids.str.upper() != "TOTAL")].copy()
    for c in ("OriginRaw", "DestRaw"):
        df[c] = df[c].astype("string").str.strip()
    df["StatusRaw"] = (df.StatusRaw.astype("string")
                       .str.replace(r"\s+", " ", regex=True)
                       .str.strip().str.title())
    if name == "rail":
        df["ShipmentID"] = "RL-" + df.ShipmentID.astype(int).astype(str)
        df["WeightKG"] = df.WeightTons * 1000
        df = df.drop(columns=["WeightTons"])
        df["ShipDate"] = pd.to_datetime(df.ShipDate)
    else:
        df["ShipmentID"] = df.ShipmentID.astype("string").str.strip().str.upper()
        fmt = "%d/%m/%Y" if name == "marine" else "%Y-%m-%d"
        df["ShipDate"] = pd.to_datetime(df.ShipDate, format=fmt)
    df["WeightKG"] = df.WeightKG.astype("int64")
    df["FreightCostPKR"] = df.FreightCostPKR.astype("int64")
    df["ModeKey"] = MODE[name]
    return df[CONTRACT + ["ModeKey"]]

def alias_lookup():
    dim = pd.read_csv(DIMS / "DimCity.csv")
    out = (dim[["CityKey", "RawAliases"]]
           .assign(RawAliases=lambda d: d.RawAliases.str.split(" | ", regex=False))
           .explode("RawAliases"))
    out["RawAliases"] = out.RawAliases.str.strip()
    out = out[out.RawAliases.notna() & (out.RawAliases != "")]
    return out.rename(columns={"RawAliases": "RawSpelling"})

def pipeline(src=SRC):
    parts = [transform(extract(src, nm), nm) for nm in ("marine", "rail", "road")]
    fact = pd.concat(parts, ignore_index=True)
    appended = len(fact)
    fact = fact.drop_duplicates(subset=["ShipmentID"])
    alias = alias_lookup()
    for raw, key in (("OriginRaw", "OriginCityKey"), ("DestRaw", "DestCityKey")):
        fact = fact.merge(alias.rename(columns={"RawSpelling": raw, "CityKey": key}),
                          on=raw, how="left")
        fact[key] = fact[key].fillna(-1).astype("int64")
    return fact.reset_index(drop=True), appended, parts

# Lab steps 1-5: shape, rename, drop junk, trim, case, prefix, convert, type, mode — per source
fact, appended, parts = pipeline()
frames["Appended"] = pd.concat(parts, ignore_index=True)
frames["PublishedFact"] = fact
marine, rail, road = parts
print(f"per-source after transform: marine {len(marine)}, rail {len(rail)}, road {len(road)}")
print(f"appended: {appended} (37 - blank - TOTAL) | loaded: {len(fact)} (appended - RD-505 duplicate)")
print(f"rail subtotal before append: {rail.WeightKG.sum():,} kg (tons x 1,000, check 2)")
print(f"contract columns: {list(fact.columns[:7])}")
assert appended == 35 and len(fact) == 34 and list(fact.columns[:7]) == CONTRACT

per-source after transform: marine 12, rail 10, road 13
appended: 35 (37 - blank - TOTAL) | loaded: 34 (appended - RD-505 duplicate)
rail subtotal before append: 295,250 kg (tons x 1,000, check 2)
contract columns: ['ShipmentID', 'ShipDate', 'OriginRaw', 'DestRaw', 'WeightKG', 'FreightCostPKR', 'StatusRaw']


**Analysis.** Steps 1–5 replay in pandas exactly as the guide's M code: rail's two title rows are skipped before headers are promoted, road's blank line and TOTAL footer are removed as two separate named decisions, all three files are renamed into the seven-name contract before the append, and the rail tons conversion happens per source so the rail subtotal is verifiable on its own before anything is combined.

In [3]:

from pathlib import Path
import pandas as pd

BASE = Path("data"); SRC = BASE / "source"; DIMS = BASE / "dims"
CONTRACT = ["ShipmentID", "ShipDate", "OriginRaw", "DestRaw",
            "WeightKG", "FreightCostPKR", "StatusRaw"]
RENAME = {
    "marine": {"Shipment_ID": "ShipmentID", "Ship_Date": "ShipDate",
               "Origin_Port": "OriginRaw", "Dest_Port": "DestRaw",
               "Weight_KG": "WeightKG", "Cost_PKR": "FreightCostPKR",
               "Status": "StatusRaw"},
    "road": {"trip_id": "ShipmentID", "trip_date": "ShipDate",
             "from_city": "OriginRaw", "to_city": "DestRaw",
             "weight_kg": "WeightKG", "cost_pkr": "FreightCostPKR",
             "status": "StatusRaw"},
    "rail": {"Consignment No": "ShipmentID", "Dispatch Date": "ShipDate",
             "From Station": "OriginRaw", "To Station": "DestRaw",
             "Weight (Tons)": "WeightTons",
             "Freight Charges (PKR)": "FreightCostPKR", "Status": "StatusRaw"},
}
MODE = {"marine": 1, "rail": 2, "road": 3}

def extract(src, name):
    if name == "rail":
        return pd.read_excel(src / "rail_shipments.xlsx", sheet_name="Rail", skiprows=2)
    return pd.read_csv(src / f"{name}_shipments.csv")

def transform(df, name):
    df = df.rename(columns=RENAME[name])
    ids = df.ShipmentID.astype("string").str.strip()
    df = df[ids.notna() & (ids != "") & (ids.str.upper() != "TOTAL")].copy()
    for c in ("OriginRaw", "DestRaw"):
        df[c] = df[c].astype("string").str.strip()
    df["StatusRaw"] = (df.StatusRaw.astype("string")
                       .str.replace(r"\s+", " ", regex=True)
                       .str.strip().str.title())
    if name == "rail":
        df["ShipmentID"] = "RL-" + df.ShipmentID.astype(int).astype(str)
        df["WeightKG"] = df.WeightTons * 1000
        df = df.drop(columns=["WeightTons"])
        df["ShipDate"] = pd.to_datetime(df.ShipDate)
    else:
        df["ShipmentID"] = df.ShipmentID.astype("string").str.strip().str.upper()
        fmt = "%d/%m/%Y" if name == "marine" else "%Y-%m-%d"
        df["ShipDate"] = pd.to_datetime(df.ShipDate, format=fmt)
    df["WeightKG"] = df.WeightKG.astype("int64")
    df["FreightCostPKR"] = df.FreightCostPKR.astype("int64")
    df["ModeKey"] = MODE[name]
    return df[CONTRACT + ["ModeKey"]]

def alias_lookup():
    dim = pd.read_csv(DIMS / "DimCity.csv")
    out = (dim[["CityKey", "RawAliases"]]
           .assign(RawAliases=lambda d: d.RawAliases.str.split(" | ", regex=False))
           .explode("RawAliases"))
    out["RawAliases"] = out.RawAliases.str.strip()
    out = out[out.RawAliases.notna() & (out.RawAliases != "")]
    return out.rename(columns={"RawAliases": "RawSpelling"})

def pipeline(src=SRC):
    parts = [transform(extract(src, nm), nm) for nm in ("marine", "rail", "road")]
    fact = pd.concat(parts, ignore_index=True)
    appended = len(fact)
    fact = fact.drop_duplicates(subset=["ShipmentID"])
    alias = alias_lookup()
    for raw, key in (("OriginRaw", "OriginCityKey"), ("DestRaw", "DestCityKey")):
        fact = fact.merge(alias.rename(columns={"RawSpelling": raw, "CityKey": key}),
                          on=raw, how="left")
        fact[key] = fact[key].fillna(-1).astype("int64")
    return fact.reset_index(drop=True), appended, parts

# Lab step 6: the alias lookup, built from DimCity — then checked against the given file
alias = alias_lookup()
given = pd.read_csv("data/alias_lookup.csv")
print(f"built alias lookup: {len(alias)} rows, {alias.CityKey.nunique()} distinct keys")
print(f"given alias_lookup.csv: {len(given)} rows, {given.CityKey.nunique()} distinct keys")
print("built == given (same spellings, same keys):",
      set(zip(alias.RawSpelling, alias.CityKey)) == set(zip(given.RawSpelling, given.CityKey)))
assert len(alias) == 23 and alias.CityKey.nunique() == 14

built alias lookup: 23 rows, 14 distinct keys
given alias_lookup.csv: 23 rows, 14 distinct keys
built == given (same spellings, same keys): True


**Analysis.** The lookup is built, not imported: DimCity's RawAliases split into rows gives 24 rows, and filtering out the blank the Unknown row leaves behind leaves 23 — the exact count the given file confirms. Skipping that filter would leave a blank spelling in the lookup, and every blank city name in a future file would then silently match it and become −1 without ever being counted.

In [4]:

from pathlib import Path
import pandas as pd

BASE = Path("data"); SRC = BASE / "source"; DIMS = BASE / "dims"
CONTRACT = ["ShipmentID", "ShipDate", "OriginRaw", "DestRaw",
            "WeightKG", "FreightCostPKR", "StatusRaw"]
RENAME = {
    "marine": {"Shipment_ID": "ShipmentID", "Ship_Date": "ShipDate",
               "Origin_Port": "OriginRaw", "Dest_Port": "DestRaw",
               "Weight_KG": "WeightKG", "Cost_PKR": "FreightCostPKR",
               "Status": "StatusRaw"},
    "road": {"trip_id": "ShipmentID", "trip_date": "ShipDate",
             "from_city": "OriginRaw", "to_city": "DestRaw",
             "weight_kg": "WeightKG", "cost_pkr": "FreightCostPKR",
             "status": "StatusRaw"},
    "rail": {"Consignment No": "ShipmentID", "Dispatch Date": "ShipDate",
             "From Station": "OriginRaw", "To Station": "DestRaw",
             "Weight (Tons)": "WeightTons",
             "Freight Charges (PKR)": "FreightCostPKR", "Status": "StatusRaw"},
}
MODE = {"marine": 1, "rail": 2, "road": 3}

def extract(src, name):
    if name == "rail":
        return pd.read_excel(src / "rail_shipments.xlsx", sheet_name="Rail", skiprows=2)
    return pd.read_csv(src / f"{name}_shipments.csv")

def transform(df, name):
    df = df.rename(columns=RENAME[name])
    ids = df.ShipmentID.astype("string").str.strip()
    df = df[ids.notna() & (ids != "") & (ids.str.upper() != "TOTAL")].copy()
    for c in ("OriginRaw", "DestRaw"):
        df[c] = df[c].astype("string").str.strip()
    df["StatusRaw"] = (df.StatusRaw.astype("string")
                       .str.replace(r"\s+", " ", regex=True)
                       .str.strip().str.title())
    if name == "rail":
        df["ShipmentID"] = "RL-" + df.ShipmentID.astype(int).astype(str)
        df["WeightKG"] = df.WeightTons * 1000
        df = df.drop(columns=["WeightTons"])
        df["ShipDate"] = pd.to_datetime(df.ShipDate)
    else:
        df["ShipmentID"] = df.ShipmentID.astype("string").str.strip().str.upper()
        fmt = "%d/%m/%Y" if name == "marine" else "%Y-%m-%d"
        df["ShipDate"] = pd.to_datetime(df.ShipDate, format=fmt)
    df["WeightKG"] = df.WeightKG.astype("int64")
    df["FreightCostPKR"] = df.FreightCostPKR.astype("int64")
    df["ModeKey"] = MODE[name]
    return df[CONTRACT + ["ModeKey"]]

def alias_lookup():
    dim = pd.read_csv(DIMS / "DimCity.csv")
    out = (dim[["CityKey", "RawAliases"]]
           .assign(RawAliases=lambda d: d.RawAliases.str.split(" | ", regex=False))
           .explode("RawAliases"))
    out["RawAliases"] = out.RawAliases.str.strip()
    out = out[out.RawAliases.notna() & (out.RawAliases != "")]
    return out.rename(columns={"RawAliases": "RawSpelling"})

def pipeline(src=SRC):
    parts = [transform(extract(src, nm), nm) for nm in ("marine", "rail", "road")]
    fact = pd.concat(parts, ignore_index=True)
    appended = len(fact)
    fact = fact.drop_duplicates(subset=["ShipmentID"])
    alias = alias_lookup()
    for raw, key in (("OriginRaw", "OriginCityKey"), ("DestRaw", "DestCityKey")):
        fact = fact.merge(alias.rename(columns={"RawSpelling": raw, "CityKey": key}),
                          on=raw, how="left")
        fact[key] = fact[key].fillna(-1).astype("int64")
    return fact.reset_index(drop=True), appended, parts

# Lab step 7-8: the eight checks as assertions, and the -1 count as a PRINT, never an assert
fact, appended, parts = pipeline()
assert list(fact.columns[:7]) == CONTRACT
assert fact[fact.ModeKey == 2].WeightKG.sum() == 295_250
assert fact.StatusRaw.nunique() == 3
assert set(fact.ShipDate.dt.month) == {2} and fact.ShipDate.dt.day.max() == 13
assert len(fact) == 34
assert fact.WeightKG.sum() == 578_650 and fact.FreightCostPKR.sum() == 4_808_000
minus_one = int(((fact.OriginCityKey == -1) | (fact.DestCityKey == -1)).sum())
print("all eight checks PASS")
print(f"rows routed to Unknown: {minus_one}  (printed and written down, not asserted — see T7)")

all eight checks PASS
rows routed to Unknown: 0  (printed and written down, not asserted — see T7)


**Analysis.** All eight checks pass as assertions, in the guide's order, with the month set written as `set(...) == {2}` rather than comparing an array — the array form raises a ValueError precisely when the locale bug it is meant to catch is present, sending you debugging the assertion instead of the dates. Check 6 stays a print on purpose: the right answer is not a constant, and an assertion would fail the whole run the first time a new city spelling appears.

In [5]:

from pathlib import Path
import pandas as pd

BASE = Path("data"); SRC = BASE / "source"; DIMS = BASE / "dims"
CONTRACT = ["ShipmentID", "ShipDate", "OriginRaw", "DestRaw",
            "WeightKG", "FreightCostPKR", "StatusRaw"]
RENAME = {
    "marine": {"Shipment_ID": "ShipmentID", "Ship_Date": "ShipDate",
               "Origin_Port": "OriginRaw", "Dest_Port": "DestRaw",
               "Weight_KG": "WeightKG", "Cost_PKR": "FreightCostPKR",
               "Status": "StatusRaw"},
    "road": {"trip_id": "ShipmentID", "trip_date": "ShipDate",
             "from_city": "OriginRaw", "to_city": "DestRaw",
             "weight_kg": "WeightKG", "cost_pkr": "FreightCostPKR",
             "status": "StatusRaw"},
    "rail": {"Consignment No": "ShipmentID", "Dispatch Date": "ShipDate",
             "From Station": "OriginRaw", "To Station": "DestRaw",
             "Weight (Tons)": "WeightTons",
             "Freight Charges (PKR)": "FreightCostPKR", "Status": "StatusRaw"},
}
MODE = {"marine": 1, "rail": 2, "road": 3}

def extract(src, name):
    if name == "rail":
        return pd.read_excel(src / "rail_shipments.xlsx", sheet_name="Rail", skiprows=2)
    return pd.read_csv(src / f"{name}_shipments.csv")

def transform(df, name):
    df = df.rename(columns=RENAME[name])
    ids = df.ShipmentID.astype("string").str.strip()
    df = df[ids.notna() & (ids != "") & (ids.str.upper() != "TOTAL")].copy()
    for c in ("OriginRaw", "DestRaw"):
        df[c] = df[c].astype("string").str.strip()
    df["StatusRaw"] = (df.StatusRaw.astype("string")
                       .str.replace(r"\s+", " ", regex=True)
                       .str.strip().str.title())
    if name == "rail":
        df["ShipmentID"] = "RL-" + df.ShipmentID.astype(int).astype(str)
        df["WeightKG"] = df.WeightTons * 1000
        df = df.drop(columns=["WeightTons"])
        df["ShipDate"] = pd.to_datetime(df.ShipDate)
    else:
        df["ShipmentID"] = df.ShipmentID.astype("string").str.strip().str.upper()
        fmt = "%d/%m/%Y" if name == "marine" else "%Y-%m-%d"
        df["ShipDate"] = pd.to_datetime(df.ShipDate, format=fmt)
    df["WeightKG"] = df.WeightKG.astype("int64")
    df["FreightCostPKR"] = df.FreightCostPKR.astype("int64")
    df["ModeKey"] = MODE[name]
    return df[CONTRACT + ["ModeKey"]]

def alias_lookup():
    dim = pd.read_csv(DIMS / "DimCity.csv")
    out = (dim[["CityKey", "RawAliases"]]
           .assign(RawAliases=lambda d: d.RawAliases.str.split(" | ", regex=False))
           .explode("RawAliases"))
    out["RawAliases"] = out.RawAliases.str.strip()
    out = out[out.RawAliases.notna() & (out.RawAliases != "")]
    return out.rename(columns={"RawAliases": "RawSpelling"})

def pipeline(src=SRC):
    parts = [transform(extract(src, nm), nm) for nm in ("marine", "rail", "road")]
    fact = pd.concat(parts, ignore_index=True)
    appended = len(fact)
    fact = fact.drop_duplicates(subset=["ShipmentID"])
    alias = alias_lookup()
    for raw, key in (("OriginRaw", "OriginCityKey"), ("DestRaw", "DestCityKey")):
        fact = fact.merge(alias.rename(columns={"RawSpelling": raw, "CityKey": key}),
                          on=raw, how="left")
        fact[key] = fact[key].fillna(-1).astype("int64")
    return fact.reset_index(drop=True), appended, parts

# Lab step 8: the refresh test — one changed constant, nothing else
night2, appended2, _ = pipeline(Path("data/night2"))
assert len(night2) == 35 and night2.WeightKG.sum() == 584_950
assert night2.FreightCostPKR.sum() == 4_860_000 and night2.StatusRaw.nunique() == 3
assert int((night2.StatusRaw == "Delayed").sum()) == 5
unresolved = sorted(set(night2.loc[night2.OriginCityKey == -1, "OriginRaw"])
                    | set(night2.loc[night2.DestCityKey == -1, "DestRaw"]))
print(f"night 2: {len(night2)} rows, {night2.WeightKG.sum():,} kg, {night2.FreightCostPKR.sum():,} PKR")
print(f"rows to Unknown: {len(unresolved)}  {unresolved}")
print("refresh PASS — nothing needed fixing by hand")

night 2: 35 rows, 584,950 kg, 4,860,000 PKR


rows to Unknown: 1  ['Chaman Border']
refresh PASS — nothing needed fixing by hand


**Analysis.** The refresh test changes one constant — the source folder — and nothing else, yet every per-source step replays correctly on a file the pipeline has never seen: the trailing-space RD-508 copy is deduplicated, the padded DELAYED value is normalised, and the Chaman Border row is kept on −1 instead of dropped. That replay, not the night-1 table, is the deliverable: it is the difference between an answer and a pipeline.

In [6]:

from pathlib import Path
import pandas as pd

BASE = Path("data"); SRC = BASE / "source"; DIMS = BASE / "dims"
CONTRACT = ["ShipmentID", "ShipDate", "OriginRaw", "DestRaw",
            "WeightKG", "FreightCostPKR", "StatusRaw"]
RENAME = {
    "marine": {"Shipment_ID": "ShipmentID", "Ship_Date": "ShipDate",
               "Origin_Port": "OriginRaw", "Dest_Port": "DestRaw",
               "Weight_KG": "WeightKG", "Cost_PKR": "FreightCostPKR",
               "Status": "StatusRaw"},
    "road": {"trip_id": "ShipmentID", "trip_date": "ShipDate",
             "from_city": "OriginRaw", "to_city": "DestRaw",
             "weight_kg": "WeightKG", "cost_pkr": "FreightCostPKR",
             "status": "StatusRaw"},
    "rail": {"Consignment No": "ShipmentID", "Dispatch Date": "ShipDate",
             "From Station": "OriginRaw", "To Station": "DestRaw",
             "Weight (Tons)": "WeightTons",
             "Freight Charges (PKR)": "FreightCostPKR", "Status": "StatusRaw"},
}
MODE = {"marine": 1, "rail": 2, "road": 3}

def extract(src, name):
    if name == "rail":
        return pd.read_excel(src / "rail_shipments.xlsx", sheet_name="Rail", skiprows=2)
    return pd.read_csv(src / f"{name}_shipments.csv")

def transform(df, name):
    df = df.rename(columns=RENAME[name])
    ids = df.ShipmentID.astype("string").str.strip()
    df = df[ids.notna() & (ids != "") & (ids.str.upper() != "TOTAL")].copy()
    for c in ("OriginRaw", "DestRaw"):
        df[c] = df[c].astype("string").str.strip()
    df["StatusRaw"] = (df.StatusRaw.astype("string")
                       .str.replace(r"\s+", " ", regex=True)
                       .str.strip().str.title())
    if name == "rail":
        df["ShipmentID"] = "RL-" + df.ShipmentID.astype(int).astype(str)
        df["WeightKG"] = df.WeightTons * 1000
        df = df.drop(columns=["WeightTons"])
        df["ShipDate"] = pd.to_datetime(df.ShipDate)
    else:
        df["ShipmentID"] = df.ShipmentID.astype("string").str.strip().str.upper()
        fmt = "%d/%m/%Y" if name == "marine" else "%Y-%m-%d"
        df["ShipDate"] = pd.to_datetime(df.ShipDate, format=fmt)
    df["WeightKG"] = df.WeightKG.astype("int64")
    df["FreightCostPKR"] = df.FreightCostPKR.astype("int64")
    df["ModeKey"] = MODE[name]
    return df[CONTRACT + ["ModeKey"]]

def alias_lookup():
    dim = pd.read_csv(DIMS / "DimCity.csv")
    out = (dim[["CityKey", "RawAliases"]]
           .assign(RawAliases=lambda d: d.RawAliases.str.split(" | ", regex=False))
           .explode("RawAliases"))
    out["RawAliases"] = out.RawAliases.str.strip()
    out = out[out.RawAliases.notna() & (out.RawAliases != "")]
    return out.rename(columns={"RawAliases": "RawSpelling"})

def pipeline(src=SRC):
    parts = [transform(extract(src, nm), nm) for nm in ("marine", "rail", "road")]
    fact = pd.concat(parts, ignore_index=True)
    appended = len(fact)
    fact = fact.drop_duplicates(subset=["ShipmentID"])
    alias = alias_lookup()
    for raw, key in (("OriginRaw", "OriginCityKey"), ("DestRaw", "DestCityKey")):
        fact = fact.merge(alias.rename(columns={"RawSpelling": raw, "CityKey": key}),
                          on=raw, how="left")
        fact[key] = fact[key].fillna(-1).astype("int64")
    return fact.reset_index(drop=True), appended, parts

# Lab step 9 (homework, unmarked): DateKey, StatusKey, RouteKey — the keyed fact for the model
fact, _, _ = pipeline()
status = pd.read_csv(DIMS / "DimStatus.csv")[["StatusKey", "StatusName"]]
route = pd.read_csv(DIMS / "DimRoute.csv")[["RouteKey", "OriginCityKey", "DestCityKey"]]
keyed = fact.copy()
keyed["DateKey"] = keyed.ShipDate.dt.year * 10000 + keyed.ShipDate.dt.month * 100 + keyed.ShipDate.dt.day
keyed = keyed.merge(status.rename(columns={"StatusName": "StatusRaw"}), on="StatusRaw", how="left")
keyed["StatusKey"] = keyed.StatusKey.fillna(-1).astype("int64")
keyed = keyed.merge(route, on=["OriginCityKey", "DestCityKey"], how="left")
keyed["RouteKey"] = keyed.RouteKey.fillna(-1).astype("int64")
keyed["ShipmentCount"] = 1
print(f"keyed fact: {len(keyed)} rows | DateKey range {keyed.DateKey.min()}-{keyed.DateKey.max()}")
print(f"StatusKey values: {sorted(keyed.StatusKey.unique())} | RouteKey -1 rows: {int((keyed.RouteKey == -1).sum())}")
print(keyed[["ShipmentID", "DateKey", "OriginCityKey", "DestCityKey", "RouteKey", "ModeKey",
             "StatusKey", "WeightKG", "FreightCostPKR"]].head(5).to_string(index=False))
assert len(keyed) == 34 and keyed.WeightKG.sum() == 578650

keyed fact: 34 rows | DateKey range 20260201-20260213
StatusKey values: [1, 2, 3] | RouteKey -1 rows: 0
ShipmentID  DateKey  OriginCityKey  DestCityKey  RouteKey  ModeKey  StatusKey  WeightKG  FreightCostPKR
   MR-1001 20260201              1            3         1        1          1     18000          142000
   MR-1002 20260202              1            7         2        1          1     22500          171000
   MR-1003 20260203              1            3         1        1          2     15000          119000
   MR-1004 20260204              2            6         3        1          1     20000          188000
   MR-1005 20260205              1            4         4        1          1     16500          128000


**Analysis.** Step 9 completes the star's keys without changing a single total: DateKey is arithmetic on the typed date (20260205, not a lossy merge), StatusKey comes from the same left-outer-then-−1 pattern as the city keys, and RouteKey is resolved on the origin/destination key pair. The keyed frame is what the .pbix model is built from, and its totals are re-asserted here so the model can never drift from the pipeline.